**MGMT298D: Science and Strategy of AI**

# Week 5: Convolutional Filters

# 1. Setup

#### We'll use a pretrained VGG16 from `keras` and visualize what its convolutional layers detect at different depths. We grab a sample image from the web to run through the network.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input, decode_predictions
from tensorflow.keras.preprocessing import image
from tensorflow.keras.models import Model

#### Upload an image and load the pretrained VGG16 model with ImageNet weights. The image gets resized to 224×224 to match what VGG16 expects.

In [ ]:
# Upload your image
from google.colab import files
uploaded = files.upload()
img_path = list(uploaded.keys())[0]

# Load and preprocess for VGG16 (224x224, BGR, centered)
img = image.load_img(img_path, target_size=(224, 224))
img_array = image.img_to_array(img)
img_batch = preprocess_input(np.expand_dims(img_array, axis=0))

# Load pretrained VGG16
model = VGG16(weights='imagenet')

# Show the image
plt.figure(figsize=(4, 4))
plt.imshow(img)
plt.title('Input Image (224×224)', fontsize=11)
plt.axis('off')
plt.show()

#### Confirm the model classifies this image correctly before we start inspecting its internals.

In [ ]:
preds = model.predict(img_batch, verbose=0)
top3 = decode_predictions(preds, top=3)[0]
for label, name, prob in top3:
    print(f'{name:>25s}: {prob:.1%}')

#### Print the layer names so we can pick which ones to visualize. VGG16 has 5 convolutional blocks, each with 2–3 conv layers followed by max pooling.

In [ ]:
for i, layer in enumerate(model.layers):
    print(f'{i:2d}  {layer.name:20s}  output: {layer.output_shape}')

---

# 2. Feature Maps

## 2.1 Extracting Activations

#### We build a new model that outputs the activations at each convolutional layer. This lets us pass the image through once and capture what every layer produces — from simple edges up to complex object parts.

In [ ]:
# Pick all Conv2D layers
conv_layers = [l for l in model.layers if 'conv' in l.name]
conv_names = [l.name for l in conv_layers]

# Build a model that outputs all conv activations at once
activation_model = Model(inputs=model.input, outputs=[l.output for l in conv_layers])
activations = activation_model.predict(img_batch, verbose=0)

print(f'{len(conv_layers)} convolutional layers extracted')
for name, act in zip(conv_names, activations):
    print(f'  {name:16s}  shape: {act.shape[1]}×{act.shape[2]}, {act.shape[3]} filters')

## 2.2 Early Layers

#### The first convolutional layer (block1_conv1) has 64 filters. Each filter responds to a different low-level pattern — horizontal edges, vertical edges, color gradients, etc. These are the building blocks everything else is built from.

In [ ]:
# Show first 16 feature maps from layer 1
act1 = activations[0][0]  # shape: (224, 224, 64)

fig, axes = plt.subplots(4, 4, figsize=(10, 10))
fig.suptitle('Block 1, Conv 1 — Edge Detectors (first 16 of 64 filters)', fontsize=13, y=1.01)
for i, ax in enumerate(axes.flat):
    ax.imshow(act1[:, :, i], cmap='viridis')
    ax.set_title(f'Filter {i}', fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 2.3 Middle Layers

#### By block 3, the network combines edges into more complex patterns — curves, textures, repeating structures. Notice the spatial resolution has shrunk from 224 to 56 because of max pooling.

In [ ]:
# block3_conv3 — middle of the network
idx_mid = conv_names.index('block3_conv3')
act_mid = activations[idx_mid][0]

fig, axes = plt.subplots(4, 4, figsize=(10, 10))
fig.suptitle(f'Block 3, Conv 3 — Texture/Pattern Detectors ({act_mid.shape[2]} filters, {act_mid.shape[0]}×{act_mid.shape[1]} spatial)', fontsize=13, y=1.01)
for i, ax in enumerate(axes.flat):
    ax.imshow(act_mid[:, :, i], cmap='viridis')
    ax.set_title(f'Filter {i}', fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 2.4 Deep Layers

#### The last convolutional layer (block5_conv3) has 512 filters at 14×14 spatial resolution. Each filter responds to high-level concepts — eyes, fur texture, snout shape. Most filters are blank for a given image because they're specialized for patterns that aren't present.

In [ ]:
# block5_conv3 — deepest conv layer
idx_deep = conv_names.index('block5_conv3')
act_deep = activations[idx_deep][0]

fig, axes = plt.subplots(4, 4, figsize=(10, 10))
fig.suptitle(f'Block 5, Conv 3 — High-Level Detectors ({act_deep.shape[2]} filters, {act_deep.shape[0]}×{act_deep.shape[1]} spatial)', fontsize=13, y=1.01)
for i, ax in enumerate(axes.flat):
    ax.imshow(act_deep[:, :, i], cmap='viridis')
    ax.set_title(f'Filter {i}', fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

---

# 3. All Layers at a Glance

#### This is the key visual — one representative filter from each convolutional layer, arranged left to right. You can see how the representation goes from pixel-level edges to abstract object-level patterns, and how the spatial size shrinks at each pooling boundary.

In [ ]:
# Pick one filter from each layer that has high activation (most "interesting")
fig, axes = plt.subplots(2, 7, figsize=(18, 5))
fig.suptitle('One Filter Per Layer — From Edges to Objects', fontsize=14, y=1.03)

for i, (name, act) in enumerate(zip(conv_names[:13], activations[:13])):
    row, col = i // 7, i % 7
    fmap = act[0]
    # Pick the filter with the highest mean activation
    best_filter = fmap.mean(axis=(0, 1)).argmax()
    axes[row, col].imshow(fmap[:, :, best_filter], cmap='inferno')
    axes[row, col].set_title(f'{name}\n{fmap.shape[0]}×{fmap.shape[1]}', fontsize=8)
    axes[row, col].axis('off')

# Hide any unused subplot
if len(conv_names) < 14:
    axes[1, 6].axis('off')

plt.tight_layout()
plt.show()

---

# 4. Filter Weights

#### We can also look at the raw filter weights (the 3×3 kernels) learned by the first layer. These are the actual numbers that get multiplied with the input — you'll recognize familiar patterns like horizontal, vertical, and diagonal edge detectors.

In [ ]:
# Get the first conv layer's weights
filters, biases = model.layers[1].get_weights()  # shape: (3, 3, 3, 64)
print(f'Filter shape: {filters.shape}  →  64 filters, each 3×3×3 (RGB)')

# Normalize to [0, 1] for display
f_min, f_max = filters.min(), filters.max()
filters_norm = (filters - f_min) / (f_max - f_min)

fig, axes = plt.subplots(4, 8, figsize=(14, 7))
fig.suptitle('First Layer: All 64 Learned 3×3 Filters (RGB)', fontsize=13, y=1.01)
for i, ax in enumerate(axes.flat):
    if i < 64:
        ax.imshow(filters_norm[:, :, :, i])
        ax.set_title(f'{i}', fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()

---

# 5. Max Pooling

#### Max pooling reduces spatial dimensions by keeping only the strongest activation in each 2×2 window. Here we show the same feature map before and after each pooling step — the detail fades but the structure survives.

In [ ]:
# Extract activations right before and after each max pool
pool_pairs = [
    ('block1_conv2', 'block1_pool'),
    ('block2_conv2', 'block2_pool'),
    ('block3_conv3', 'block3_pool'),
]

pool_model = Model(
    inputs=model.input,
    outputs=[model.get_layer(n).output for pair in pool_pairs for n in pair]
)
pool_acts = pool_model.predict(img_batch, verbose=0)

fig, axes = plt.subplots(3, 2, figsize=(10, 12))
fig.suptitle('Before vs After Max Pooling — Same Filter', fontsize=14, y=1.01)

for row, (before_name, after_name) in enumerate(pool_pairs):
    before = pool_acts[row * 2][0]
    after = pool_acts[row * 2 + 1][0]
    filt = before.mean(axis=(0, 1)).argmax()  # pick strongest filter

    axes[row, 0].imshow(before[:, :, filt], cmap='viridis')
    axes[row, 0].set_title(f'{before_name} ({before.shape[0]}×{before.shape[1]})', fontsize=10)
    axes[row, 0].axis('off')

    axes[row, 1].imshow(after[:, :, filt], cmap='viridis')
    axes[row, 1].set_title(f'{after_name} ({after.shape[0]}×{after.shape[1]})', fontsize=10)
    axes[row, 1].axis('off')

plt.tight_layout()
plt.show()